# Lesson 12.4 - LLM Application Deployment, Observability & Governance (telemetry & evaluation demo)

This notebook simulates an LLM production operations layer:

- request traces,
- quality and safety telemetry,
- SLO and policy alerts,
- governance checklist output.


## Objectives

1. Build trace records for LLM requests.
2. Compute observability metrics and SLO checks.
3. Generate governance-oriented operational reports.

In [1]:
from __future__ import annotations

from dataclasses import dataclass, asdict
from typing import List

import numpy as np
import pandas as pd

np.random.seed(101)

## Step 1: Simulate LLM Request Traces

In [2]:
@dataclass
class TraceEvent:
    request_id: str
    model: str
    prompt_version: str
    latency_ms: float
    input_tokens: int
    output_tokens: int
    grounded_score: float
    safety_flag: int
    http_status: int


def generate_trace(i: int) -> TraceEvent:
    latency = np.random.normal(loc=650, scale=120)
    in_tok = int(max(50, np.random.normal(420, 80)))
    out_tok = int(max(20, np.random.normal(180, 50)))
    grounded = float(np.clip(np.random.normal(0.84, 0.12), 0, 1))
    safety = int(np.random.rand() < 0.04)
    status = 500 if np.random.rand() < 0.02 else 200
    return TraceEvent(
        request_id=f"req_{i:04d}",
        model="gpt-4o-mini-like",
        prompt_version="v3.2.1",
        latency_ms=float(max(100, latency)),
        input_tokens=in_tok,
        output_tokens=out_tok,
        grounded_score=grounded,
        safety_flag=safety,
        http_status=status,
    )


traces: List[TraceEvent] = [generate_trace(i) for i in range(1, 401)]
trace_df = pd.DataFrame([asdict(t) for t in traces])
trace_df.head()

,request_id,model,prompt_version,latency_ms,input_tokens,output_tokens,grounded_score,safety_flag,http_status
0,req_0001,gpt-4o-mini-like,v3.2.1,974.821981,470,225,0.900459,0,200
1,req_0002,gpt-4o-mini-like,v3.2.1,548.230762,468,79,0.928815,0,200
2,req_0003,gpt-4o-mini-like,v3.2.1,672.643437,359,133,0.954607,0,200
3,req_0004,gpt-4o-mini-like,v3.2.1,962.716074,474,195,1.000000,0,200
4,req_0005,gpt-4o-mini-like,v3.2.1,445.269688,327,173,0.886863,0,200


## Step 2: Compute Observability Metrics

In [3]:
metrics = {
    "requests": int(len(trace_df)),
    "error_rate": float((trace_df["http_status"] != 200).mean()),
    "p95_latency_ms": float(trace_df["latency_ms"].quantile(0.95)),
    "avg_total_tokens": float((trace_df["input_tokens"] + trace_df["output_tokens"]).mean()),
    "avg_grounded_score": float(trace_df["grounded_score"].mean()),
    "safety_flag_rate": float(trace_df["safety_flag"].mean()),
}
metrics

{'requests': 400,
 'error_rate': 0.0125,
 'p95_latency_ms': 856.5792185266623,
 'avg_total_tokens': 605.0725,
 'avg_grounded_score': 0.8279607463152214,
 'safety_flag_rate': 0.035}

## Step 3: SLO / Policy Alert Checks

In [4]:
SLOS = {
    "max_error_rate": 0.03,
    "max_p95_latency_ms": 900,
    "min_grounded_score": 0.80,
    "max_safety_flag_rate": 0.05,
}

alerts = []
if metrics["error_rate"] > SLOS["max_error_rate"]:
    alerts.append("CRITICAL: error rate above SLO")
if metrics["p95_latency_ms"] > SLOS["max_p95_latency_ms"]:
    alerts.append("WARN: p95 latency above SLO")
if metrics["avg_grounded_score"] < SLOS["min_grounded_score"]:
    alerts.append("CRITICAL: groundedness below threshold")
if metrics["safety_flag_rate"] > SLOS["max_safety_flag_rate"]:
    alerts.append("CRITICAL: safety flag rate above threshold")

alerts if alerts else ["No active alerts"]

['No active alerts']

## Step 4: Governance Checklist Report

In [5]:
governance_checklist = pd.DataFrame(
    [
        {"control": "Prompt/version logging", "status": "PASS", "evidence": "prompt_version in every trace"},
        {"control": "Token usage tracking", "status": "PASS", "evidence": "input/output tokens logged"},
        {"control": "Safety monitoring", "status": "PASS", "evidence": "safety_flag field + rate alerts"},
        {"control": "SLO policy", "status": "PASS", "evidence": "latency/error/groundedness checks"},
        {"control": "Incident escalation path", "status": "Planned", "evidence": "Define on-call runbook owner"},
    ]
)

governance_checklist

,control,status,evidence
0,Prompt/version logging,PASS,prompt_version in every trace
1,Token usage tracking,PASS,input/output tokens logged
2,Safety monitoring,PASS,safety_flag field + rate alerts
3,SLO policy,PASS,latency/error/groundedness checks
4,Incident escalation path,Planned,Define on-call runbook owner


## Production Case Studies & Exceptions

### Case 1: RAG assistant with hidden quality regression
- Infrastructure metrics looked healthy, but groundedness dropped after retrieval change.
- Added quality metrics to release gates, not just latency/error gates.

### Case 2: LLM API outage in peak traffic
- Primary provider throttled requests.
- Fallback model and queue-based degraded mode preserved core functionality.

### Case 3 (Exception): Internal analytics chatbot
- Lower strictness on response polish was acceptable.
- Team prioritized cost controls over maximal fluency quality.

## Interview Questions & Answers

1. **Q: What should an LLM trace include?**
   **A:** Prompt version, model, latency, token usage, retrieval metadata, and safety signals.

2. **Q: Why is groundedness metric important?**
   **A:** It measures factual support and helps detect hallucination risk.

3. **Q: How do you define LLM SLOs?**
   **A:** Set thresholds for reliability, latency, quality, and safety indicators.

4. **Q: What triggers rollback in LLM deployments?**
   **A:** Sustained SLO or policy breaches compared to baseline.

5. **Q: How is LLM observability different from standard API observability?**
   **A:** It includes semantic quality and safety metrics, not just infrastructure health.

6. **Q: Why track prompt versions in production?**
   **A:** To correlate behavior changes with prompt updates.

7. **Q: How do you manage LLM operational cost?**
   **A:** Monitor token usage, set budgets, and route by task complexity.

8. **Q: What governance artifacts should be maintained?**
   **A:** Risk register, model/system card, change log, and incident reports.

9. **Q: How do you reduce safety incidents?**
   **A:** Guardrails, adversarial testing, and policy-based filtering.

10. **Q: What is a practical first step to LLM governance maturity?**
   **A:** Establish consistent logging, ownership, and periodic risk review.
